In [55]:
import re
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [56]:
resume_text = """
John Doe
Machine Learning Engineer

Skills:
Python, SQL, Pandas, NumPy, Scikit-learn,
Machine Learning, Deep Learning, TensorFlow,
NLP, Git, Docker

Experience:
Built machine learning models for classification and regression.
Worked with data preprocessing, feature engineering and model evaluation.
Developed NLP applications and deployed models using Docker.
"""

job_description = """
We are looking for a Machine Learning Engineer with experience in
Python, SQL, Pandas, NumPy, Scikit-learn, Machine Learning,
Deep Learning, TensorFlow, NLP, Git, Docker and AWS.

The candidate should be comfortable with data preprocessing,
feature engineering, model evaluation and deploying machine learning models.
"""

In [57]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [58]:
clean_resume = clean_text(resume_text)
clean_job = clean_text(job_description)

print("Cleaned Resume:")
print(clean_resume)
print("\nCleaned Job Description:")
print(clean_job)

Cleaned Resume:
john doe machine learning engineer skills python sql pandas numpy scikit-learn machine learning deep learning tensorflow nlp git docker experience built machine learning models for classification and regression worked with data preprocessing feature engineering and model evaluation developed nlp applications and deployed models using docker

Cleaned Job Description:
we are looking for a machine learning engineer with experience in python sql pandas numpy scikit-learn machine learning deep learning tensorflow nlp git docker and aws the candidate should be comfortable with data preprocessing feature engineering model evaluation and deploying machine learning models


In [59]:
SKILLS = [
    "python",
    "java",
    "c++",
    "c#",
    "javascript",
    "typescript",

    "sql",
    "mysql",
    "postgresql",
    "mongodb",

    "pandas",
    "numpy",
    "scikit-learn",


    "machine learning",
    "deep learning",
    "tensorflow",
    "pytorch",
    "keras",

    "nlp",
    "natural language processing",
    "computer vision",

    "matplotlib",
    "seaborn",
    "power bi",
    "tableau",


    "git",
    "github",
    "docker",
    "kubernetes",


    "aws",
    "azure",
    "gcp",

    "spark",
    "hadoop",

    "flask",
    "fastapi",
    "streamlit",
    "django"
]

### SKILL Extractor Function

In [60]:
def extract_skills(text, skills = SKILLS):
    text = clean_text(text)

    found_skills = []

    for skill in skills:
        pattern = r'(?<![a-z0-9])' + re.escape(skill) + r'(?![a-z0-9])'
        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills

In [61]:
extract_skills(resume_text,SKILLS)

['python',
 'sql',
 'pandas',
 'numpy',
 'scikit-learn',
 'machine learning',
 'deep learning',
 'tensorflow',
 'nlp',
 'git',
 'docker']

In [62]:
resume_skills = extract_skills(resume_text)

print("Resume Skills:")
print(resume_skills)


Resume Skills:
['python', 'sql', 'pandas', 'numpy', 'scikit-learn', 'machine learning', 'deep learning', 'tensorflow', 'nlp', 'git', 'docker']


In [63]:
job_required_skills = extract_skills(job_description, SKILLS)

print("Required skills for the job:")
print(job_required_skills)

Required skills for the job:
['python', 'sql', 'pandas', 'numpy', 'scikit-learn', 'machine learning', 'deep learning', 'tensorflow', 'nlp', 'git', 'docker', 'aws']


### Skill Matching Function

In [64]:
def match_skills(resume_skills, job_required_skills):

    resume_set = set(resume_skills)
    job_set = set(job_required_skills)

    matched = sorted(resume_set.intersection(job_set))
    missing = sorted(job_set.difference(resume_set))

    if len(job_set) == 0:
        score = 0
    else:
        score = len(matched) / len(job_set) * 100

    return matched, missing, score

### Calculate Skill Match Score

In [65]:
matched_skills, missing_skills, skill_match_score =  match_skills(
    resume_skills, job_required_skills
)

print("Matched Skills:")
print(matched_skills)

print("\n Missing Skills:")
print(missing_skills)

print(f"\n skill Match Score: {skill_match_score:.2f}%")

Matched Skills:
['deep learning', 'docker', 'git', 'machine learning', 'nlp', 'numpy', 'pandas', 'python', 'scikit-learn', 'sql', 'tensorflow']

 Missing Skills:
['aws']

 skill Match Score: 91.67%


### Loading Transformner

In [66]:
model = SentenceTransformer('all-MiniLM-L6-V2')
print("Model Loaded Successfully")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3548.36it/s]


Model Loaded Successfully


### Generate Resume Embedding

In [67]:
resume_embedding = model.encode([clean_resume])

print("Resume embedding shape:")
print(resume_embedding.shape)

Resume embedding shape:
(1, 384)


### Generating Job Embedding

In [68]:
job_embedding = model.encode([clean_job])
print("Joob Embedding Shape:")
print(job_embedding.shape)

Joob Embedding Shape:
(1, 384)


### calculate Semantic Similarity


In [69]:
semantic_similarity = cosine_similarity(
    resume_embedding,
    job_embedding
)[0][0]

semantic_score = semantic_similarity * 100

print(f"Semantic Similarity Score: {semantic_score:.2f}%")

Semantic Similarity Score: 84.85%


In [70]:
SKILL_WEIGHT = 0.60
SEMANTNIC_WEIGHT = 0.40

### Calculating Overall Match

In [71]:
overall_score = (
    skill_match_score * SKILL_WEIGHT + semantic_score * SEMANTNIC_WEIGHT
)
print(f"Overall Score: {overall_score:.2f}%")

Overall Score: 88.94%


In [72]:
def get_recommendations(score):
    if score >= 80:
        return "Strong Match" 
    elif score >= 65:
        return "Good Match" 
    elif score >= 50:
        return "Fair Match"
    else:
        return "Not a Match"
    

In [73]:
recommendation = get_recommendations(overall_score)

print(f"Recommendations:", recommendation)

Recommendations: Strong Match


In [74]:
analysis_results = {
    "matched_skills": round(skill_match_score, 2),
    "semantic_similarity_score": round(semantic_score,2),
    "overall_match_score": round(overall_score, 2),
    "matched_skills" : matched_skills,
    "missing_skills": missing_skills,
    "recommendation": recommendation
}

analysis_results

{'matched_skills': ['deep learning',
  'docker',
  'git',
  'machine learning',
  'nlp',
  'numpy',
  'pandas',
  'python',
  'scikit-learn',
  'sql',
  'tensorflow'],
 'semantic_similarity_score': np.float32(84.85),
 'overall_match_score': np.float32(88.94),
 'missing_skills': ['aws'],
 'recommendation': 'Strong Match'}

In [75]:
print("=" * 50)
print( " AI RESUME ANALYZER")
print("=" * 50)

print(f"skill Match Score  : {skill_match_score:.2f}%")
print(f"Semantic Similarity : {semantic_score:.2f}%")
print(f"Overall Match Score : {overall_score:.2f}%")
print(f"recommendation      ; {recommendation}")


print("\n Matched Skills:")
for skill in matched_skills:
    print(f"{skill}")

print("\n Missing Skills:")
if missing_skills:
    for skill in missing_skills:
        print(f" {skill}")
else:
    print("None")

print("=" * 50)

 AI RESUME ANALYZER
skill Match Score  : 91.67%
Semantic Similarity : 84.85%
Overall Match Score : 88.94%
recommendation      ; Strong Match

 Matched Skills:
deep learning
docker
git
machine learning
nlp
numpy
pandas
python
scikit-learn
sql
tensorflow

 Missing Skills:
 aws


In [76]:
test_job = """
Data Analyst required with Python, SQL, Pandas, NumPy,
Power BI, Tableau, Excel and Machine Learning.
"""

test_job_skills = extract_skills(test_job)
print("Test Job Skills:")
print(test_job_skills)

Test Job Skills:
['python', 'sql', 'pandas', 'numpy', 'machine learning', 'power bi', 'tableau']


In [77]:
test_matched, test_missing, test_skill_score = match_skills(
    resume_skills,
    test_job_skills
)


print("Matched Skills:")
print(test_matched)

print("\n Missing SKills:")
print(test_missing)

print(f"\n Skill Match Score: {test_skill_score:.2f}%")


Matched Skills:
['machine learning', 'numpy', 'pandas', 'python', 'sql']

 Missing SKills:
['power bi', 'tableau']

 Skill Match Score: 71.43%


In [78]:
test_job_clean = clean_text(test_job)
test_job_embedding = model.encode([test_job_clean])

test_similarity = cosine_similarity(
    resume_embedding,
    test_job_embedding
)[0][0]

test_semantic_score = test_similarity * 100
print(f"Semantic Similarity: {test_semantic_score:.2f}%")

Semantic Similarity: 47.84%


In [81]:
test_overall_score = (
    test_skill_score * SKILL_WEIGHT + test_semantic_score * SEMANTNIC_WEIGHT
)

print(f"OVerall Match Score: {test_overall_score:.2f}%")
print(f"recommendation: {get_recommendations(test_overall_score)}")

OVerall Match Score: 61.99%
recommendation: Fair Match


In [84]:
jobs = {
    "Machine Learning Engineer": job_description,

    "Data Analyst": test_job,

    "Python Developer": """
    Python developer required with Python, Django, Flask,
    FastAPI, SQL, Git, Docker and REST API experience.
    """
}

### Rank Jobs

In [ ]:
results = []

for job_title,jd in jobs.items():
    jd_clean = clean_text(jd)

    #Extract Job Skills
    jd_skills = extract_skills(jd_clean)

    #skill matching
    matched, missing, skill_score = match_skills(
        resume_skills,
        jd_skills
    )

    # Semantic Similarity
    jd_embedding = model.encode([jd_clean])

    similarity = cosine_similarity(
        resume_embedding,
        jd_embedding
    )[0][0]

    semantic_score = similarity * 100

    #Overall Score
    overall = (
        skill_score * SKILL_WEIGHT + semantic_score * SEMANTNIC_WEIGHT
    )

    results.append({
        "Job": job_title,
        "Skill Match %": round(skill_score, 2),
        "Semantic Similarity %": round(semantic_score,2),
        "Overall Match %": round(overall,2),
        "Recommendations": get_recommendations(overall),
        "Matched Skills": ",".join(matched),
        "Missing Skills":",".join(missing)

    })

In [86]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Overall Match %",
    ascending = False
).reset_index(drop=True)

results_df

,Job,Skill Match %,Semantic Similarity %,Overall Match %,Recommendations,Matched Skills,Missing Skills
0,Machine Learning Engineer,91.67,84.849998,7254.140137,Strong Match,"deep learning,docker,git,machine learning,nlp,...",aws
1,Data Analyst,71.43,47.840000,2331.179932,Strong Match,"machine learning,numpy,pandas,python,sql","power bi,tableau"
2,Python Developer,57.14,44.450001,2010.319946,Strong Match,"docker,git,python,sql","django,fastapi,flask"
